In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn
from torch import optim
import torch.nn.functional as F
import numpy as np
import glob
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(context='talk', style='ticks',
        color_codes=True, rc={'legend.frameon': False})
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
plt.rcParams["font.family"] = 'Arial'
plt.rcParams.update({'font.size': 24})
plt.rcParams['svg.fonttype'] = 'none'

In [2]:
RANDOM_SEED = 42
torch.backends.cudnn.deterministic = True
generator = torch.Generator().manual_seed(RANDOM_SEED)

In [ ]:
class MordredDataset(Dataset):
    def __init__(self, data_dir):
        pass
    def __len__(self):
        pass
    def __getitem__(self, idx):
        return (data, labels)

In [3]:
def get_splits(dataset_len, splits = [0.8, 0.1, 0.1]):
    training_len = int(dataset_len * splits[0])
    valid_len = int(dataset_len * splits[1])
    test_len = dataset_len - (training_len + valid_len)
    return([training_len, valid_len, test_len])

In [ ]:
esm_dataset = ESMDataset('/home/sabari/ProteinSol/topoformer/data/soluprotgeom/processed/train/esm_embeddings')
batch_size = 1000
shuffle_train = True
pin_memory = True
num_workers = 8

In [ ]:
split_lens = get_splits(esm_dataset.__len__())
train_set, val_set, test_set = random_split(esm_dataset, split_lens, generator=generator)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=shuffle_train, pin_memory=pin_memory, num_workers=num_workers)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)

In [ ]:
class ESMANNModel(nn.Module):
    def __init__(self,
                n_layers: int,
                input_size: int,
                output_size: int = 1,
                hidden_dim_size: int = 256):
        layers = []
        layers.append(nn.Linear(input_size, hidden_dim_size))
        layers.append(nn.ReLU())
        for layer in n_layers:
            layers.append(nn.Linear(hidden_dim_size, hidden_dim_size))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_dim_size, output_size))
        self.net = nn.Sequential([*layers])
        
    def forward(self, x: torch.Tensor):
        out = self.net(x)
        return out

In [ ]:
def train_epoch(
    train_loader: DataLoader,
    model: nn.Module,
    optimizer: optim.Optimizer,
):
    l1_running_loss = 0
    bce_running_loss = 0
    for batch_idx, sample in enumerate(train_loader):
        optimizer.zero_grad()
        preds = model(sample[0])
        preds = nn.Sigmoid(preds)
        l1_loss = F.l1_loss(preds, sample[1])
        bce_loss = F.bce_loss(preds, sample[1])
        bce_loss.backward()
        optimizer.step()
        l1_running_loss += l1_loss.item()
        bce_running_loss += l2_loss.item()
        # if per_batch_log_interval:
        #     if batch_idx % per_batch_log_interval == 0:
        #         wandb.log({"per_batch_train_l1_loss": l1_running_loss})
        #         wandb.log({"per_batch_train_l2_loss": l2_running_loss})
    return (l1_running_loss / len(train_loader), bce_running_loss / len(train_loader))

In [ ]:
def train_nn(train_loader,
            valid_loader, 
            num_epochs,
            model,
            optimizer):

    for epoch in tqdm(range(num_epochs)):
        model.train()
        l1_loss, bce_loss = train_epoch(train_loader, model, optimizer)
        for batch_idx, sample in enumerate(valid_loader):
            with model.eval():
                valid_preds = model(sample[0])
                valids_preds = nn.Sigmoid(preds)
                valid_l1_loss = F.l1_loss(preds, sample[1])
                valid_bce_loss = F.bce_loss(preds, sample[1])
            
        print(f"Epoch no: {epoch} --- Train BCE Loss: {bce_loss} \tValid BCE Loss: {valid_bce_loss} \tTrain L1 Loss: {l1_loss} \tValid L1 Loss: {valid_l1_loss}")
        

In [ ]:
model = ESMANNModel(n_layers = 5, input_size = 1280)
optimizer = optim.Adamax(model.parameters(), lr=lr)
train_nn(train_loader = train_loader , valid_loader = valid_loader, num_epochs = 200, model, optimzer, )